# Moonshot AI Kimi K3 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Kimi K3 is the newest generation of Moonshot AI's Kimi models. Unlike the K2 models in
[`01-kimi-k2`](01-kimi-k2.ipynb), it is on `bedrock-runtime` only, behind an inference
profile, and it serves both Chat Completions and the Responses API there.

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Kimi K3 | — | `us.moonshotai.kimi-k3` | Inference profile only; the prefix follows the Region's geography, with `global.` where the geography has no profile (section 1) |

IDs as of October 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region. The setup cell is the same as in
`01-kimi-k2`, so it also prints the K2 IDs.

**Covered:** the inference profile and its prefix, Chat Completions and the Responses API
on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check, the `bedrock-runtime` ID for a model, and a lenient JSON parser.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient, runtime_id_for

REGION = "us-east-1"
K25 = "moonshotai.kimi-k2.5"
THINKING = "moonshotai.kimi-k2-thinking"
K3 = "moonshotai.kimi-k3"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

# The two endpoints do not agree on these IDs, so print the ID each one takes rather
# than a yes/no: K2 Thinking drops the "ai" from its vendor prefix on bedrock-runtime,
# and K3 is profile-only there. A bedrock-mantle ID sent to bedrock-runtime is refused
# outright, so resolve the runtime ID instead of reusing the string.
for model in (K25, THINKING, K3,):
    served = endpoints_for(model, REGION)
    mantle_id = model if served["mantle"] else "-"
    runtime_id = runtime_id_for(model, REGION) or "-"
    print(f"{model:29} mantle: {mantle_id:29} runtime: {runtime_id}")

moonshotai.kimi-k2.5          mantle: moonshotai.kimi-k2.5          runtime: moonshotai.kimi-k2.5
moonshotai.kimi-k2-thinking   mantle: moonshotai.kimi-k2-thinking   runtime: moonshot.kimi-k2-thinking


moonshotai.kimi-k3            mantle: -                             runtime: us.moonshotai.kimi-k3


## 1. Calling K3 through its inference profile

K3 is on `bedrock-runtime` only, and it is `INFERENCE_PROFILE`-only: the bare ID
`moonshotai.kimi-k3` is refused with `Invocation of model ID moonshotai.kimi-k3 with
on-demand throughput isn’t supported`. Call the inference profile instead,
`us.moonshotai.kimi-k3` in `us-east-1`. The prefix follows the Region's geography rather
than being either `us.` or `global.`: on 7 October 2026 `ap-south-1` and `ap-south-2`
carried `in.moonshotai.kimi-k3` alongside `global.moonshotai.kimi-k3` and no `us.` one,
and `eu-central-1` carried `global.moonshotai.kimi-k3` alone. Prefer the geographic
profile where the Region has one, because it names the Regions it routes to:
`in.moonshotai.kimi-k3` lists `ap-south-1` and `ap-south-2`, while the `global.` profile
lists a model ARN with no Region and so does not say. `resolve_runtime_id()` in the
helper module returns the form your Region carries, preferring a geographic profile over
`global.`.

Unlike the K2 models, K3 serves both Chat Completions and the Responses API, on the
OpenAI-compatible path of `bedrock-runtime`. The OpenAI SDK reaches it with the same
short-term API key as before, pointed at a different base URL.

In [2]:
K3_PROFILE = "us.moonshotai.kimi-k3"
k3 = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

chat = k3.chat.completions.create(
    model=K3_PROFILE,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
print("Chat Completions:", (chat.choices[0].message.content or "").strip())

reply = k3.responses.create(
    model=K3_PROFILE,
    input="What is the capital of Australia? Answer in one word.",
    max_output_tokens=1000,
    store=False,
)
print("Responses API   :", reply.output_text.strip())

Chat Completions: Canberra


Responses API   : Canberra


## Things that differ

What trips people up on Kimi K3, and what to do about it. Behaviour observed in
September and October 2026.

| Behaviour | What you see | What to do |
|---|---|---|
| K3 with its bare ID | `ValidationException … on-demand throughput isn’t supported` | Call the inference profile (section 1). |
| K3 with a prefix the Region does not carry | 400 `The provided model identifier is invalid.` | The prefix follows the geography: `us.moonshotai.kimi-k3` fails this way in `ap-south-1` and `in.moonshotai.kimi-k3` fails this way in `us-east-1`, 3 of 3 each on 7 October 2026. Look the profile up (section 1). |
| K3 and `top_p` | 400 `This model accepts 'top_p' only with the value 0.95.` | Leave `top_p` unset for K3. |
| K3 sampling on Converse | `ValidationException: This model doesn't support the temperature field` | Leave `temperature` and `topP` out of `inferenceConfig` for K3. |
| Prompt caching on Converse | `cachePoint` refused, as `ValidationException` or `AccessDeniedException`, as it is for the K2 models | The `AccessDeniedException` is not a permissions problem. K3 does cache on its OpenAI-compatible paths. |

## Next

- The K2 models, on both endpoints:
  [`08-moonshot-kimi/01`](01-kimi-k2.ipynb)
- Inference profiles and Converse on `bedrock-runtime`:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)